In [ ]:
# Diffusion model dependencies (TabDDPM + CoDi)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# CoDi: ChaejeongLee/CoDi (_vendor/CoDi)
%pip install -q ForestDiffusion xgboost category-encoders libzero rtdl imbalanced-learn absl-py tensorboardX

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "goggle" / "src"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "CoDi"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_codi


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
covertype = fetch_ucirepo(id=31)

# data (as pandas dataframes)
X = covertype.data.features
y = covertype.data.targets

# metadata
print(covertype.metadata)

# variable information
print(covertype.variables)

covertype_data = pd.concat([X, y], axis=1)

# ----------------------------------------------------
# Target column handling
# ----------------------------------------------------
# Ensure Cover_Type is the explicit target variable
if "Cover_Type" in covertype_data.columns:
    target_col = "Cover_Type"
elif "cover_type" in covertype_data.columns:
    covertype_data = covertype_data.rename(columns={"cover_type": "Cover_Type"})
    target_col = "Cover_Type"
else:
    target_col = covertype_data.columns[-1]
    covertype_data = covertype_data.rename(columns={target_col: "Cover_Type"})
    target_col = "Cover_Type"

# Keep full dataset for repeatable subsampling when experiment settings is re-run.
covertype_data_full = covertype_data.copy()
print(f"Full dataset cached: {covertype_data_full.shape}")


In [ ]:
# ----------------------------------------------------
# Preprocess features before synthetic data generation
# ----------------------------------------------------

X = covertype_data_full.drop(columns=[target_col]).copy()
y = covertype_data_full[target_col].copy()

covertype_data_full = pd.concat([X, y], axis=1)
covertype_data = covertype_data_full.copy()

print(f'Target variable: {target_col}')
print(f'Dataset shape after preprocessing: {covertype_data_full.shape}')


In [ ]:
# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000          # random real samples drawn from full dataset
TEST_SIZE = 0.2           # 20% holdout for unseen TSTR evaluation
SEED = 42

# Speed controls (set DEV_MODE=False, FAST_MODE=False for full paper run)
FAST_MODE = True
DEV_MODE = True
RUN_QUALITY_EVAL = True

N_SYNTH_SAMPLES = 200 if DEV_MODE else 1000

_epoch_fast = 2 if DEV_MODE else 5
TabDDPM_EPOCHS = 50
WGAN_EPOCHS = 50 if FAST_MODE else 100
SDV_EPOCHS = _epoch_fast if FAST_MODE else 300

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

ALL_GENERATORS = [
    "CTGAN", "CopulaGAN", "TVAE", "GaussianCopula", "CoDi", "TabDDPM"
]
GENERATORS_TO_EVAL = ALL_GENERATORS

# All 6 generators use exactly these 10 cartographic features + Cover_Type target.
CONTINUOUS_COLS = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]

GENERATOR_FEATURE_COLS = CONTINUOUS_COLS.copy()
if len(covertype_data_full) < N_SAMPLES:
    raise RuntimeError(
        f"Full dataset has only {len(covertype_data_full)} rows. "
        "Re-run the data loading and preprocessing cells first."
    )

covertype_data, _ = train_test_split(
    covertype_data_full,
    train_size=N_SAMPLES,
    stratify=covertype_data_full[target_col],
    random_state=SEED,
)
covertype_data = covertype_data.reset_index(drop=True)

# 80% for generator training, 20% held out unseen for TSTR evaluation.
train_real, test_real = train_test_split(
    covertype_data,
    test_size=TEST_SIZE,
    stratify=covertype_data[target_col],
    random_state=SEED,
)
train_real = train_real.reset_index(drop=True)
test_real = test_real.reset_index(drop=True)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(train_real)

GENERATOR_FEATURE_COLS = [c for c in GENERATOR_FEATURE_COLS if c in train_real.columns]
if len(GENERATOR_FEATURE_COLS) != 10:
    raise ValueError(
        f"Expected 10 shared generator features, found {len(GENERATOR_FEATURE_COLS)}: "
        f"{GENERATOR_FEATURE_COLS}"
    )

GEN_COLS = GENERATOR_FEATURE_COLS + [target_col]
train_gen = train_real[GEN_COLS].copy()
test_gen = test_real[GEN_COLS].copy()

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_gen)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []

print(f'Random subsample: {covertype_data.shape}')
print(f'Generator training set (shared 10 features): {train_gen.shape}')
print(f'Holdout test set (shared 10 features): {test_gen.shape}')
print(f'DEV_MODE: {DEV_MODE} | FAST_MODE: {FAST_MODE} | quality eval: {RUN_QUALITY_EVAL}')
print(f'Generators enabled: {GENERATORS_TO_EVAL}')
print(f'Synthetic samples per generator: {N_SYNTH_SAMPLES}')
print(f'All 6 generators use these 10 features + target: {GEN_COLS}')


In [ ]:
# ---------------------------------------------------
# SINGLE RUN — setup + TabDDPM
# ---------------------------------------------------
seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SYNTH_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')


In [ ]:
# CoDi
if 'CoDi' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training CoDi...')
        synthetic_codi = train_codi(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SYNTH_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['CoDi'] = synthetic_codi.copy()
        print('CoDi: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_codi,
                metadata=train_metadata,
            )
            scores['CoDi'] = quality.get_score()
            print('CoDi:', round(scores['CoDi'], 4))
        else:
            print('CoDi: trained (quality eval skipped)')
    except Exception as e:
        print('CoDi Failed (training/sampling):')
        traceback.print_exc()
    if 'CoDi' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('CoDi: skipped (not in GENERATORS_TO_EVAL)')


In [ ]:
# ---------------------------------------------------
# SDV Models (CTGAN, CopulaGAN, TVAE, GaussianCopula)
# ---------------------------------------------------
sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "TVAE": TVAESynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata),
}

for model_name, model in sdv_models.items():
    if model_name not in GENERATORS_TO_EVAL:
        print(f"{model_name}: skipped (not in GENERATORS_TO_EVAL)")
        continue

    try:
        model.fit(train_gen)
        synthetic_data = model.sample(N_SYNTH_SAMPLES)
        synthetic_datasets[model_name] = synthetic_data.copy()

        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_gen,
                synthetic_data=synthetic_data,
                metadata=train_metadata
            )
            scores[model_name] = quality.get_score()
            print(f"{model_name}: {round(scores[model_name], 4)}")
        else:
            print(f"{model_name}: trained (quality eval skipped)")

    except Exception as e:
        print(f"{model_name} Failed: {e}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(max_iter=500, solver="liblinear", random_state=42),
        "SVM-RBF": LinearSVC(max_iter=500, dual="auto", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=12),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(max_iter=5000, solver="liblinear", random_state=42),
        "SVM-RBF": SVC(kernel="rbf", cache_size=1000, tol=1e-3, random_state=42),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd


In [ ]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
    use_holdout=False,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    def _std(values):
        return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]
            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if use_holdout:
                # Fixed holdout test; resample training rows per seed.
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )

                _, X_test, _, y_test = train_test_split(
                    X_test,
                    y_test,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_test),
                )

            clf = clone(model)
            if 'random_state' in clf.get_params():
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, average="weighted", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": _std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": _std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": _std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": _std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {_std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {_std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {_std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {_std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)


In [ ]:
import pandas as pd

label_col = "Cover_Type"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "CoDi",
    "TabDDPM"
]

seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real) — 80% train / 20% holdout")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Generators: {len(model_order)}"
)
print(f"Generator training set: {train_gen.shape} | Holdout test set: {test_gen.shape}")

trtr_results = evaluate_models(
    train_df=train_gen,
    test_df=test_gen,
    label_col=label_col,
    models=models,
    seeds=seeds,
    use_holdout=True,
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR (train on synthetic, test on 20% holdout)")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_gen,
        label_col=label_col,
        models=models,
        seeds=seeds,
        use_holdout=True,
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


In [ ]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")
